# 07: Geocoding & Address Processing

This notebook demonstrates geocoding capabilities in siege_utilities:

1. **Address to Coordinates** - Convert addresses to lat/lon
2. **Address Concatenation** - Build standardized addresses
3. **Batch Geocoding** - Process multiple addresses
4. **Country Code Filtering** - Restrict results by country

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install geopy
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import siege_utilities as su
su.configure_shared_logging(level="INFO")

from siege_utilities.geo.geocoding import (
    get_coordinates,
    concatenate_addresses,
    use_nominatim_geocoder
)

import pandas as pd
import time

su.log_info("Geocoding imports successful!")

# --- Path configuration ---
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")

## 1. Single Address Geocoding

Convert an address string to latitude/longitude coordinates.

In [ ]:
# Geocode a single address
address = "1600 Pennsylvania Avenue NW, Washington, DC"

su.log_info(f"Geocoding: {address}")
coords = get_coordinates(address, country_codes='us')

if coords:
    lat, lon = coords  # Returns tuple (lat, lon)
    su.log_info(f"Result: latitude={lat:.6f}, longitude={lon:.6f}")
else:
    su.log_warning("No results found")

In [ ]:
# More examples
test_addresses = [
    "Empire State Building, New York, NY",
    "Golden Gate Bridge, San Francisco, CA",
    "Space Needle, Seattle, WA"
]

su.log_info("Geocoding landmarks:")
su.log_info("-" * 60)
for addr in test_addresses:
    # Rate limit - Nominatim requires 1 second between requests
    time.sleep(1)
    coords = get_coordinates(addr, country_codes='us')
    if coords:
        lat, lon = coords  # Returns tuple (lat, lon)
        su.log_info(f"{addr[:40]:40} -> ({lat:.4f}, {lon:.4f})")
    else:
        su.log_warning(f"{addr[:40]:40} -> Not found")

## 2. Address Concatenation

Build standardized address strings from components.

In [ ]:
# Build address from components
address = concatenate_addresses(
    street="123 Main Street",
    city="San Francisco",
    state_province_area="California",
    postal_code="94102",
    country="USA"
)

su.log_info(f"Concatenated address: {address}")

In [ ]:
# Partial address (some fields missing)
partial_address = concatenate_addresses(
    city="Los Angeles",
    state_province_area="CA"
)

su.log_info(f"Partial address: {partial_address}")

## 3. Batch Geocoding

Geocode multiple addresses with rate limiting.

In [ ]:
# Sample addresses to geocode
addresses_df = pd.DataFrame({
    'name': ['City Hall', 'Airport', 'University'],
    'street': ['200 N Spring St', '1 World Way', '405 Hilgard Ave'],
    'city': ['Los Angeles', 'Los Angeles', 'Los Angeles'],
    'state': ['CA', 'CA', 'CA']
})

su.log_info("Sample addresses:")
su.log_info(f"\n{addresses_df}")

In [ ]:
def batch_geocode(df, street_col, city_col, state_col, delay=1.0):
    """
    Batch geocode addresses in a DataFrame.
    
    Args:
        df: DataFrame with address columns
        street_col: Column name for street address
        city_col: Column name for city
        state_col: Column name for state
        delay: Seconds between requests (Nominatim requires >= 1)
        
    Returns:
        DataFrame with lat/lon columns added
    """
    result_df = df.copy()
    result_df['latitude'] = None
    result_df['longitude'] = None
    result_df['geocode_status'] = None
    
    for idx, row in result_df.iterrows():
        # Build address
        address = concatenate_addresses(
            street=row[street_col],
            city=row[city_col],
            state_province_area=row[state_col]
        )
        
        # Geocode - returns tuple (lat, lon) or None
        coords = get_coordinates(address, country_codes='us')
        
        if coords:
            lat, lon = coords
            result_df.at[idx, 'latitude'] = lat
            result_df.at[idx, 'longitude'] = lon
            result_df.at[idx, 'geocode_status'] = 'success'
        else:
            result_df.at[idx, 'geocode_status'] = 'not_found'
        
        # Rate limit
        time.sleep(delay)
    
    return result_df

# Geocode the addresses
su.log_info("Batch geocoding (this may take a few seconds)...")
geocoded_df = batch_geocode(addresses_df, 'street', 'city', 'state')

su.log_info("Results:")
su.log_info(f"\n{geocoded_df[['name', 'latitude', 'longitude', 'geocode_status']]}")

## 4. Using Nominatim Directly

The `use_nominatim_geocoder` function provides more control.

In [ ]:
# Get detailed geocoding result
# use_nominatim_geocoder returns a JSON string with detailed info
import json

result_str = use_nominatim_geocoder(
    query_address="Statue of Liberty, New York",
    country_codes='us'
)

if result_str:
    result = json.loads(result_str)
    su.log_info("Geocoding result:")
    for key, value in result.items():
        su.log_info(f"  {key}: {value}")
else:
    su.log_warning("No results found")

## 5. Visualize Geocoded Points

Plot geocoded points on a map.

In [ ]:
import matplotlib.pyplot as plt
from siege_utilities.geo.spatial_data import get_census_boundaries

# Get California boundary for context
states = get_census_boundaries(year=2020, geographic_level='state')
ca = states[states['statefp'] == '06']

# Filter to successful geocodes
geocoded_points = geocoded_df[geocoded_df['geocode_status'] == 'success']

if len(geocoded_points) > 0:
    fig, ax = plt.subplots(figsize=(8, 10))
    
    # Plot state boundary
    ca.plot(ax=ax, facecolor='lightgray', edgecolor='black')
    
    # Plot points
    ax.scatter(
        geocoded_points['longitude'],
        geocoded_points['latitude'],
        c='red', s=100, zorder=5, marker='o'
    )
    
    # Add labels
    for idx, row in geocoded_points.iterrows():
        ax.annotate(
            row['name'],
            xy=(row['longitude'], row['latitude']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8
        )
    
    # Focus on LA area
    ax.set_xlim(-119, -117)
    ax.set_ylim(33, 35)
    ax.set_title('Geocoded Locations in Los Angeles')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    plt.tight_layout()
    plt.savefig('/tmp/geocoded_points.png', dpi=100, bbox_inches='tight')
    su.log_info("Map saved to /tmp/geocoded_points.png")
    plt.show()
else:
    su.log_warning("No geocoded points to visualize")

## 6. Geocode Validation (Dual-Mode)

The `siege_utilities.distributed.spark_utils` module provides Spark-native functions for
validating geocode data at scale. Since PySpark may not be installed in every environment,
this section uses a **dual-mode pattern**: pandas-only validation first, then the equivalent
Spark implementation when PySpark is available.

Three functions are demonstrated:

| Function | Purpose |
|----------|---------|
| `validate_geocode_data()` | Filter rows to only valid lat/lon ranges |
| `mark_valid_geocode_data()` | Add a boolean `is_valid` flag column (preserves all rows) |
| `clean_and_reorder_bbox()` | Parse bounding-box strings and reorder to `[min_lon, min_lat, max_lon, max_lat]` |

In [ ]:
# --- Dual-mode: detect PySpark availability ---
SPARK_AVAILABLE = False
spark = None

try:
    from pyspark.sql import SparkSession
    SPARK_AVAILABLE = True
    su.log_info("PySpark is installed -- Spark validation examples will run")
except ImportError:
    su.log_info("PySpark not installed -- pandas-only mode")

# If Spark is available, create a local session for demonstration
if SPARK_AVAILABLE:
    try:
        spark = SparkSession.builder \
            .master("local[*]") \
            .appName("NB07_GeoValidation") \
            .getOrCreate()
        su.log_info(f"Spark session active: {spark.version}")
    except Exception as e:
        su.log_warning(f"Could not create Spark session: {e}")
        SPARK_AVAILABLE = False

# Import the Spark geocode validation functions (only if Spark is available)
if SPARK_AVAILABLE:
    from siege_utilities.distributed.spark_utils import (
        validate_geocode_data,
        mark_valid_geocode_data,
        clean_and_reorder_bbox,
    )
    su.log_info("Spark geocode validation functions imported successfully")

In [ ]:
# --- Sample geocode data (used by all three demos) ---
# Mix of valid, out-of-range, and null coordinates
sample_geocode_data = pd.DataFrame({
    "id":        [1,     2,     3,     4,     5,     6,     7],
    "name":      ["White House", "North Pole", "Bad Lat", "Bad Lon", "Null Lat",  "Null Both", "Sydney Opera House"],
    "latitude":  [38.8977,       90.0,         -999.0,    34.0522,   None,         None,        -33.8568],
    "longitude": [-77.0365,      0.0,          -77.0,     999.0,     -118.2437,    None,        151.2153],
    "bbox":      [
        "[38.0, 39.0, -78.0, -76.0]",
        "[89.0, 90.0, -1.0, 1.0]",
        None,
        "[33.0, 35.0, 998.0, 1000.0]",
        None,
        None,
        "[-34.0, -33.0, 150.0, 152.0]",
    ],
})

su.log_info("Sample geocode data:")
su.log_info(f"\n{sample_geocode_data.to_string(index=False)}")

### 6a. `validate_geocode_data()` -- Filter to Valid Coordinates

This function **removes** rows whose latitude/longitude fall outside the valid ranges
(`-90 <= lat <= 90`, `-180 <= lon <= 180`) or are null.

**Signature:** `validate_geocode_data(df, lat_col_name: str, lon_col_name: str)`

The pandas equivalent uses simple boolean indexing.

In [ ]:
# --- 6a. validate_geocode_data() ---

# ---- Pandas path (always available) ----
def validate_geocode_data_pandas(df, lat_col, lon_col):
    """Pandas equivalent of spark_utils.validate_geocode_data().
    
    Filters out rows where lat/lon are null or outside valid ranges.
    """
    mask = (
        df[lat_col].notna()
        & df[lon_col].notna()
        & df[lat_col].between(-90, 90)
        & df[lon_col].between(-180, 180)
    )
    return df[mask].reset_index(drop=True)

valid_pd = validate_geocode_data_pandas(sample_geocode_data, "latitude", "longitude")
su.log_info(f"Pandas: {len(sample_geocode_data)} rows -> {len(valid_pd)} valid rows after filtering")
su.log_info(f"\n{valid_pd[['id', 'name', 'latitude', 'longitude']].to_string(index=False)}")

# ---- Spark path (when available) ----
if SPARK_AVAILABLE:
    su.log_info("\n--- Spark path ---")
    spark_df = spark.createDataFrame(sample_geocode_data.fillna(float("nan")))
    # Replace NaN back to null for Spark compatibility
    from pyspark.sql.functions import col, when, isnan
    for c in ["latitude", "longitude"]:
        spark_df = spark_df.withColumn(c, when(isnan(col(c)), None).otherwise(col(c)))

    valid_spark_df = validate_geocode_data(spark_df, "latitude", "longitude")
    su.log_info(f"Spark:  {spark_df.count()} rows -> {valid_spark_df.count()} valid rows after filtering")
    valid_spark_df.select("id", "name", "latitude", "longitude").show(truncate=False)
else:
    su.log_info("Spark path skipped (PySpark not available)")

### 6b. `mark_valid_geocode_data()` -- Add a Validity Flag

Unlike `validate_geocode_data()`, this function **preserves all rows** and adds a boolean
column (default name `is_valid`) indicating whether each row's coordinates are valid.

**Signature:** `mark_valid_geocode_data(df, lat_col_name: str, lon_col_name: str, output_col_name: str = "is_valid")`

In [ ]:
# --- 6b. mark_valid_geocode_data() ---

# ---- Pandas path ----
def mark_valid_geocode_data_pandas(df, lat_col, lon_col, output_col="is_valid"):
    """Pandas equivalent of spark_utils.mark_valid_geocode_data().
    
    Adds a boolean column without removing any rows.
    """
    result = df.copy()
    result[output_col] = (
        result[lat_col].notna()
        & result[lon_col].notna()
        & result[lat_col].between(-90, 90)
        & result[lon_col].between(-180, 180)
    )
    return result

marked_pd = mark_valid_geocode_data_pandas(sample_geocode_data, "latitude", "longitude")
su.log_info("Pandas: All rows preserved with 'is_valid' flag:")
su.log_info(f"\n{marked_pd[['id', 'name', 'latitude', 'longitude', 'is_valid']].to_string(index=False)}")

valid_count = marked_pd["is_valid"].sum()
invalid_count = len(marked_pd) - valid_count
su.log_info(f"\nValid: {valid_count}  |  Invalid: {invalid_count}")

# ---- Spark path ----
if SPARK_AVAILABLE:
    su.log_info("\n--- Spark path ---")
    # Re-use the spark_df created earlier (with nulls properly set)
    marked_spark_df = mark_valid_geocode_data(spark_df, "latitude", "longitude")
    su.log_info("Spark: All rows preserved with 'is_valid' flag:")
    marked_spark_df.select("id", "name", "latitude", "longitude", "is_valid").show(truncate=False)

    # Show counts by validity
    from pyspark.sql.functions import count
    marked_spark_df.groupBy("is_valid").agg(count("*").alias("row_count")).show()
else:
    su.log_info("Spark path skipped (PySpark not available)")

### 6c. `clean_and_reorder_bbox()` -- Normalize Bounding Boxes

Geocoding APIs often return bounding boxes as bracket-delimited strings in the order
`[min_lat, max_lat, min_lon, max_lon]`. Apache Sedona (and most GIS tools) expect
`[min_lon, min_lat, max_lon, max_lat]`.

`clean_and_reorder_bbox()` strips brackets, splits the string, and produces both the
individual coordinate columns and a reordered array column.

**Signature:** `clean_and_reorder_bbox(df, bbox_col)`

**Produced columns** (where `bbox_col` = `"bbox"`):
- `bbox_cleaned` -- bracket-free string
- `bbox_split` -- array of string tokens
- `bbox_min_lat`, `bbox_max_lat`, `bbox_min_lon`, `bbox_max_lon` -- individual doubles
- `bbox_reordered` -- array `[min_lon, min_lat, max_lon, max_lat]`

In [ ]:
# --- 6c. clean_and_reorder_bbox() ---

# ---- Pandas path ----
def clean_and_reorder_bbox_pandas(df, bbox_col):
    """Pandas equivalent of spark_utils.clean_and_reorder_bbox().
    
    Strips brackets, splits on comma, extracts individual lat/lon doubles,
    and builds a reordered list [min_lon, min_lat, max_lon, max_lat].
    """
    import re
    result = df.copy()

    # Strip brackets
    cleaned_col = f"{bbox_col}_cleaned"
    result[cleaned_col] = result[bbox_col].apply(
        lambda v: re.sub(r"[\[\]]", "", v) if pd.notna(v) else None
    )

    # Split into tokens
    split_col = f"{bbox_col}_split"
    result[split_col] = result[cleaned_col].apply(
        lambda v: [t.strip() for t in v.split(",")] if pd.notna(v) else None
    )

    # Extract individual coordinates (input order: min_lat, max_lat, min_lon, max_lon)
    def _safe_float(arr, idx):
        try:
            return float(arr[idx])
        except (TypeError, IndexError, ValueError):
            return None

    result[f"{bbox_col}_min_lat"] = result[split_col].apply(lambda a: _safe_float(a, 0))
    result[f"{bbox_col}_max_lat"] = result[split_col].apply(lambda a: _safe_float(a, 1))
    result[f"{bbox_col}_min_lon"] = result[split_col].apply(lambda a: _safe_float(a, 2))
    result[f"{bbox_col}_max_lon"] = result[split_col].apply(lambda a: _safe_float(a, 3))

    # Reorder to [min_lon, min_lat, max_lon, max_lat]
    reordered_col = f"{bbox_col}_reordered"
    result[reordered_col] = result.apply(
        lambda row: [
            row[f"{bbox_col}_min_lon"],
            row[f"{bbox_col}_min_lat"],
            row[f"{bbox_col}_max_lon"],
            row[f"{bbox_col}_max_lat"],
        ]
        if pd.notna(row[f"{bbox_col}_min_lat"])
        else None,
        axis=1,
    )
    return result

bbox_pd = clean_and_reorder_bbox_pandas(sample_geocode_data, "bbox")
display_cols = ["id", "name", "bbox", "bbox_min_lat", "bbox_max_lat",
                "bbox_min_lon", "bbox_max_lon", "bbox_reordered"]
su.log_info("Pandas: Bounding box cleaned and reordered:")
su.log_info(f"\n{bbox_pd[display_cols].to_string(index=False)}")

# ---- Spark path ----
if SPARK_AVAILABLE:
    su.log_info("\n--- Spark path ---")
    # Build a Spark DF with just id, name, and bbox
    bbox_spark_df = spark.createDataFrame(
        sample_geocode_data[["id", "name", "bbox"]].fillna("")
    )
    # Restore empty strings to null for bbox
    from pyspark.sql.functions import col, when, length, trim
    bbox_spark_df = bbox_spark_df.withColumn(
        "bbox", when(length(trim(col("bbox"))) == 0, None).otherwise(col("bbox"))
    )

    bbox_result_df = clean_and_reorder_bbox(bbox_spark_df, "bbox")
    su.log_info("Spark: Bounding box cleaned and reordered:")
    bbox_result_df.select(
        "id", "name", "bbox", "bbox_min_lat", "bbox_max_lat",
        "bbox_min_lon", "bbox_max_lon", "bbox_reordered"
    ).show(truncate=False)
else:
    su.log_info("Spark path skipped (PySpark not available)")

In [ ]:
# --- Cleanup: stop Spark session if we started one ---
if SPARK_AVAILABLE and spark is not None:
    spark.stop()
    su.log_info("Spark session stopped")

## Summary

| Section | Function | Purpose |
|---------|----------|---------|
| Geocoding | `get_coordinates()` | Simple address to lat/lon |
| Geocoding | `concatenate_addresses()` | Build address strings |
| Geocoding | `use_nominatim_geocoder()` | Detailed geocoding with options |
| Validation | `validate_geocode_data()` | Filter DataFrame to valid lat/lon ranges (Spark) |
| Validation | `mark_valid_geocode_data()` | Add boolean `is_valid` flag, preserving all rows (Spark) |
| Validation | `clean_and_reorder_bbox()` | Normalize bounding-box strings for Sedona/GIS tools (Spark) |

**Important Notes:**
- Nominatim (OpenStreetMap) requires 1 second delay between requests
- Use `country_codes` parameter to improve accuracy
- For high-volume geocoding, consider commercial services
- Geocode validation functions live in `siege_utilities.distributed.spark_utils`
- All Spark validation functions have pandas equivalents shown in Section 6 for environments without PySpark